In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
from pathlib import Path
import numpy as np
import torch
import pandas as pd
import open3d as o3d
import faiss
from tqdm import tqdm
from scipy.spatial.transform import Rotation as R
import MinkowskiEngine as ME
from torch.utils.data import Dataset, DataLoader
from opr.models.place_recognition import MinkLoc3Dv2, MinkLoc3D

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
2025-08-13 08:26:19.755 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


# Constants

In [3]:
REPO_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition") # For running in Docker

AIRI_DATA_DIR = Path("/home/docker_mmpr/Datasets/2024-12-14-AIRI-dataset/processed-data/slam/slam/")
# AIRI_DATA_DIR.mkdir(exist_ok=True, parents=True) # Only for local install
assert AIRI_DATA_DIR.exists(), f"Data directory {AIRI_DATA_DIR} does not exist."

PC_QUANTIZATION_SIZE = 0.05
DISTANCE_THRESHOLD = 5.0  # meters for evaluation

# DataReader definition

In [4]:
class RawPointCloudDataset(Dataset):
    def __init__(self, poses_csv_path: Path, agg_thres: float = 0):
        """
        Args:
            poses_csv_path: Path to CSV with pose information
            agg_thres: Aggregation threshold in meters (0 = no aggregation)
        """
        self.poses_df = pd.read_csv(poses_csv_path)
        self.agg_thres = agg_thres
        self.slam_maps = {
            '1': 'AIRI_slam_1',
            '2': 'AIRI_slam_2', 
            '3': 'AIRI_slam_3'
        }
        
        # Pre-load all raw point clouds and poses
        self.raw_data = []
        for map_id, group in self.poses_df.groupby(self.poses_df['lidar_ts'].str[-1]):
            slam_map = self.slam_maps[map_id]
            lidar_dir = AIRI_DATA_DIR / slam_map / 'keyframe_map' / 'scans'
            
            for _, row in tqdm(group.iterrows(), desc=f"Loading {slam_map}", total=len(group)):
                ts = int(row['lidar_ts'][:-2])  # Remove map identifier
                pcd_path = lidar_dir / f"{ts:06d}.pcd"
                
                # Load raw point cloud
                pcd = o3d.io.read_point_cloud(str(pcd_path))
                points = np.asarray(pcd.points)
                
                # Store raw data with pose
                self.raw_data.append({
                    'points': points,
                    'pose': row[['px', 'py', 'pz', 'qx', 'qy', 'qz', 'qw']].values.astype(float),
                    'lidar_ts': row['lidar_ts']
                })
    
    def __len__(self):
        return len(self.raw_data)
    
    def __getitem__(self, idx):
        if self.agg_thres > 0:
            return self._get_aggregated_item(idx)
        return self._get_single_item(idx)
    
    def _get_single_item(self, idx):
        """Get single scan without aggregation"""
        data = self.raw_data[idx]
        points = data['points']
        pose = data['pose']
        
        # Prepare features (use intensity if available, else ones)
        feats = np.ones((points.shape[0], 1))
        
        return {
            "pointcloud_lidar_coords": torch.tensor(points, dtype=torch.float32),
            "pointcloud_lidar_feats": torch.tensor(feats, dtype=torch.float32),
            "pose": torch.tensor(pose, dtype=torch.float32),
            "lidar_ts": data['lidar_ts']
        }
    
    def _get_aggregated_item(self, idx):
        """Get aggregated scans within threshold distance"""
        current_data = self.raw_data[idx]
        current_pose = current_data['pose']
        current_loc = current_pose[:3]
        
        # Find nearby scans to aggregate
        agg_points = []
        for i in range(max(0, idx-100), idx+1):  # Look back up to 100 frames
            data = self.raw_data[i]
            pose = data['pose']
            if np.linalg.norm(pose[:3] - current_loc) <= self.agg_thres:
                # Transform points to current frame
                T_current = self.pose_to_matrix(*current_pose)
                T_other = self.pose_to_matrix(*pose)
                T_other_to_current = np.linalg.inv(T_current) @ T_other
                
                points = data['points']
                points_transformed = (T_other_to_current[:3, :3] @ points.T).T + T_other_to_current[:3, 3]
                agg_points.append(points_transformed)
        
        if agg_points:
            agg_points = np.concatenate(agg_points, axis=0)
        else:
            agg_points = current_data['points']  # fallback to single scan

        agg_points = np.ascontiguousarray(agg_points)
        
        # Prepare features (use intensity if available, else ones)
        feats = np.ones((agg_points.shape[0], 1))
        
        return {
            "pointcloud_lidar_coords": torch.tensor(agg_points, dtype=torch.float32),
            "pointcloud_lidar_feats": torch.tensor(feats, dtype=torch.float32),
            "pose": torch.tensor(current_pose, dtype=torch.float32),
            "lidar_ts": current_data['lidar_ts']
        }
    
    @staticmethod
    def pose_to_matrix(tx, ty, tz, qx, qy, qz, qw):
        rot = R.from_quat([qx, qy, qz, qw]).as_matrix()
        T = np.eye(4)
        T[:3, :3] = rot
        T[:3, 3] = [tx, ty, tz]
        return T
    
    def collate_fn(self, batch):
        poses = torch.stack([item['pose'] for item in batch])
        timestamps = [item['lidar_ts'] for item in batch]
        
        coords_list = [e["pointcloud_lidar_coords"] for e in batch]
        feats_list = [e["pointcloud_lidar_feats"] for e in batch]
        
        quantized_coords_list = []
        quantized_feats_list = []
        
        for coords, feats in zip(coords_list, feats_list):
            coords = coords.contiguous()
            feats = feats.contiguous()
            
            quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                coordinates=coords,
                features=feats,
                quantization_size=PC_QUANTIZATION_SIZE,
            )
            quantized_coords_list.append(quantized_coords)
            quantized_feats_list.append(quantized_feats)
        
        return {
            "poses": poses,
            "pointclouds_lidar_coords": ME.utils.batched_coordinates(quantized_coords_list),
            "pointclouds_lidar_feats": torch.cat(quantized_feats_list),
            "lidar_timestamps": timestamps
        }

# Init model

In [5]:
# weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_baseline.pth")
weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3d_nclt.pth")
# weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_nclt.pth")

# model = MinkLoc3Dv2()
model = MinkLoc3D()
model.load_state_dict(weights, strict=False)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
else:
    print("CUDA is not available, running on CPU.")

# Evaluate model

In [6]:
def evaluate_model(model, db_loader, query_loader, agg_thres, top_k=5):
    device = next(model.parameters()).device
    
    # Extract database descriptors
    db_descriptors = []
    db_poses = []
    db_timestamps = []
    
    with torch.no_grad():
        for batch in tqdm(db_loader, desc=f"Processing database (agg_thres={agg_thres}m)"):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            descriptors = model(batch)["final_descriptor"]
            db_descriptors.append(descriptors.cpu())
            db_poses.append(batch["poses"].cpu())
            db_timestamps.extend(batch["lidar_timestamps"])
    
    db_descriptors = torch.cat(db_descriptors, dim=0).numpy()
    db_poses = torch.cat(db_poses, dim=0).numpy()
    
    # Build FAISS index
    faiss_index = faiss.IndexFlatL2(db_descriptors.shape[1])
    faiss_index.add(db_descriptors)
    
    # Process queries
    query_descriptors = []
    query_poses = []
    query_timestamps = []
    
    with torch.no_grad():
        for batch in tqdm(query_loader, desc=f"Processing queries (agg_thres={agg_thres}m)"):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            descriptors = model(batch)["final_descriptor"]
            query_descriptors.append(descriptors.cpu())
            query_poses.append(batch["poses"].cpu())
            query_timestamps.extend(batch["lidar_timestamps"])
    
    query_descriptors = torch.cat(query_descriptors, dim=0).numpy()
    query_poses = torch.cat(query_poses, dim=0).numpy()
    
    # Search for nearest neighbors
    k = top_k
    _, indices = faiss_index.search(query_descriptors, k)
    
    # Calculate metrics
    recalls = [0] * k
    trans_errors = []
    rot_errors = []
    
    for q_idx in range(len(query_poses)):
        q_pose = query_poses[q_idx]
        q_loc, q_rot = q_pose[:3], q_pose[3:]
        q_ts = query_timestamps[q_idx]
        
        best_trans_error = float('inf')
        best_rot_error = float('inf')
        match_at_k = [False] * k
        
        for rank, db_idx in enumerate(indices[q_idx]):
            db_pose = db_poses[db_idx]
            db_loc, db_rot = db_pose[:3], db_pose[3:]
            db_ts = db_timestamps[db_idx]
            
            # Skip if same timestamp (trivial match)
            if q_ts == db_ts:
                match_at_k[rank:] = [True] * (k - rank)
                best_trans_error = 0
                best_rot_error = 0
                break
                
            trans_error = np.linalg.norm(q_loc - db_loc)
            rot_error = 2 * np.arccos(np.clip(np.abs(np.dot(q_rot, db_rot)), -1.0, 1.0))
            
            best_trans_error = min(best_trans_error, trans_error)
            best_rot_error = min(best_rot_error, rot_error)
            
            if trans_error < DISTANCE_THRESHOLD:
                match_at_k[rank:] = [True] * (k - rank)
        
        trans_errors.append(best_trans_error)
        rot_errors.append(best_rot_error)
        
        for i in range(k):
            recalls[i] += match_at_k[i]
    
    # Calculate final metrics
    trans_errors = np.array(trans_errors)
    rot_errors = np.array(rot_errors)
    
    print(f"\nEvaluation Results (Aggregation Threshold = {agg_thres}m):")
    print(f"Mean translation error: {np.mean(trans_errors):.2f}m")
    print(f"Median translation error: {np.median(trans_errors):.2f}m")
    print(f"Mean rotation error: {np.rad2deg(np.mean(rot_errors)):.2f}°")
    print(f"Median rotation error: {np.rad2deg(np.median(rot_errors)):.2f}°")
    
    for i in range(k):
        print(f"Recall@{i+1}: {recalls[i]/len(query_poses):.2%}")

In [7]:
for agg_thres in [3, 4]:
        # Create datasets
        db_dataset = RawPointCloudDataset(AIRI_DATA_DIR / 'db_lidar_poses_full.csv', agg_thres=agg_thres)
        query_dataset = RawPointCloudDataset(AIRI_DATA_DIR / 'query_lidar_poses_full.csv', agg_thres=agg_thres)
        
        # Create dataloaders
        db_loader = DataLoader(
            db_dataset,
            batch_size=16,
            shuffle=False,
            collate_fn=db_dataset.collate_fn,
            num_workers=4,
            pin_memory=True
        )
        
        query_loader = DataLoader(
            query_dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=query_dataset.collate_fn,
            num_workers=4,
            pin_memory=True
        )
        
        # Run evaluation
        evaluate_model(model, db_loader, query_loader, agg_thres)

Processing queries (agg_thres=3m): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 485/485 [00:28<00:00, 16.88it/s]



Evaluation Results (Aggregation Threshold = 3m):
Mean translation error: 4.00m
Median translation error: 1.60m
Mean rotation error: 128.58°
Median rotation error: 160.84°
Recall@1: 73.20%
Recall@2: 81.86%
Recall@3: 84.95%
Recall@4: 86.60%
Recall@5: 87.22%


Processing queries (agg_thres=4m): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 485/485 [00:39<00:00, 12.25it/s]


Evaluation Results (Aggregation Threshold = 4m):
Mean translation error: 3.49m
Median translation error: 1.81m
Mean rotation error: 124.13°
Median rotation error: 157.78°
Recall@1: 83.71%
Recall@2: 88.66%
Recall@3: 90.93%
Recall@4: 91.55%
Recall@5: 91.75%
